In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

from diffpy.mpdf import *

import maginteractions

from diffpy.structure import loadStructure

%matplotlib notebook

In [2]:
# build magstruc with primitive unit cell

triclinic = False

structure_file = 'NaMnO2Data/fitStruc_triShortRange_005K.cif'

NaMnO2 = loadStructure(structure_file)

if triclinic:
    a = NaMnO2.lattice.a
    b = NaMnO2.lattice.b
    c = NaMnO2.lattice.c
    alpha = NaMnO2.lattice.alpha
    beta = NaMnO2.lattice.beta
    gamma = NaMnO2.lattice.gamma
else:
    a = b = 3.17167
    c = 5.79900
    alpha = beta = 110.4993
    gamma = 53.5987
    NaMnO2.lattice.a = a
    NaMnO2.lattice.b = b
    NaMnO2.lattice.c = c
    NaMnO2.lattice.alpha = alpha
    NaMnO2.lattice.beta = beta
    NaMnO2.lattice.gamma = gamma
    
print(NaMnO2)

lattice=Lattice(a=3.17167, b=3.17167, c=5.799, alpha=110.499, beta=110.499, gamma=53.5987)
Na   0.500000 0.500000 0.500000 1.0000
Mn   0.000000 0.000000 0.000000 1.0000
O    0.709026 0.704427 0.204309 1.0000
O    0.290974 0.295573 0.795691 1.0000


In [3]:
mspec = MagSpecies()
mspec.struc = NaMnO2
mspec.label = 'Mn3+'
mspec.strucIdxs = [0]
mspec.basisvecs = np.array([[1.0, 1.0, 0]])
mspec.origin = np.array([0, 0, 0])
mspec.ffparamkey = 'Mn3'

magstruc = MagStructure()
magstruc.loadSpecies(mspec)
magstruc.makeAll()

In [4]:
# populate spins with mcif

mcif = 'NaMnO2Data/1.409_NaMnO2.mcif'
mstruc0 = create_from_mcif(mcif, ffparamkey='Mn2')
mstruc0.makeAll()

# rotate mcif spins/atoms to match current magstruc orientation
theta = -1.0571653032274788
rotation_matrix = np.array([[np.cos(theta), -np.sin(theta), 0],
                           [np.sin(theta), np.cos(theta), 0],
                           [0, 0, 1]])

magstruc.spins = mstruc0.spins @ rotation_matrix
magstruc.atoms = mstruc0.atoms @ rotation_matrix
magstruc.calcIdxs = mstruc0.calcIdxs

MagStructure creation from mcif file successful.


In [5]:
distances = np.apply_along_axis(np.linalg.norm, 1, magstruc.atoms)
viewmask = distances < 6
magstruc.visualize(magstruc.atoms[viewmask], magstruc.spins[viewmask], showcrystalaxes=True)

<IPython.core.display.Javascript object>

In [6]:
### import data

high_idx = 6 # index where paramagnet data starts

[data_temps, xi_data, xi_err] = np.loadtxt('NaMnO2Data/mPDFresults_NaMnO2_xi-vs-T.txt',
                                           delimiter = ' ', skiprows = 2).T
[data_temps_high, xi_high_data, xi_high_err] = np.loadtxt('NaMnO2Data/mPDFresults_NaMnO2_xi-vs-T.txt',
                                                          delimiter = ' ', skiprows = 2)[high_idx:].T

[_, lmop_data, lmop_err] = np.loadtxt('NaMnO2Data/mPDFresults_NaMnO2_m-vs-T.txt',
                                      delimiter = ' ', skiprows = 2).T
[_, lmop_high_data, lmop_high_err] = np.loadtxt('NaMnO2Data/mPDFresults_NaMnO2_m-vs-T.txt',
                                                delimiter = ' ', skiprows = 2)[high_idx:].T

lmop_data = 2 * np.sqrt(lmop_data)
lmop_err = 0.5 * lmop_data * lmop_err / (lmop_data / 2) ** 2

lmop_high_data = 2 * np.sqrt(lmop_high_data)
lmop_high_err = 0.5 * lmop_high_data * lmop_high_err / (lmop_high_data / 2) ** 2

# Previously published values

In [7]:
temps = data_temps # np.linspace(0.1,600.0,20)

In [8]:
j_ij_zorko = -65.0 * np.array([1, 0.44]) # Zorko 2008
j_ij_dally = np.array([-6.16, -0.77])/8.617333262e-2 # Dally 2018

j_ij_wadhwa_1 = np.array([-11.3,-3.26])/8.617333262e-2 # Wadhwa 2025
j_ij_wadhwa_2 = np.array([-3.87,-0.82])/8.617333262e-2
j_ij_wadhwa_3 = np.array([-6.37,-0.65])/8.617333262e-2
j_ij_wadhwa_4 = np.array([-6.1,-0.55])/8.617333262e-2

j_ijs = [j_ij_zorko, j_ij_dally, j_ij_wadhwa_1, j_ij_wadhwa_2, j_ij_wadhwa_3, j_ij_wadhwa_4]
j_ij_names = ["Ref. [38]", "Ref. [39]", "Wadhwa 1", "Wadhwa 2", "Wadhwa 3", "Wadhwa 4"]

In [9]:
# Calculate J(q)

r_max = 5.0
spin_squared = 4

j_ij_test = j_ij_zorko
magint_test = maginteractions.MagInteractions(magstruc = magstruc, temperatures = temps, j_ij = j_ij_test,
                                              spin_squared = spin_squared, r_max = r_max)
magint_test.make_interaction_matrix()
magint_test.calculate_orf_reciprocal_space(n_bz_vecs = 4)

orfs_reciprocal_space = magint_test.orfs.copy()

/home/edison/Documents/Projects/MagInteractions/maginteractions.py:225: ComplexWarning: Casting complex values to real discards the imaginary part
  u_q[k] = eig_vecs.astype('float64')


In [10]:
# visualize J(q)

import matplotlib.colors as mcolors

qxs, qys, qzs = magint_test.q_vecs.T

vmin = 100
vmax = 142.25

qa = 2 * np.pi / a
qb = 2 * np.pi / b
qc = 2 * np.pi / c


fig, ax = plt.subplots(subplot_kw = {"projection": "3d"})
norm = mcolors.Normalize(vmin = vmin, vmax = vmax)
j_q_plot = ax.scatter(qxs, qys, qzs, c = magint_test.j_q[:,-1], norm = norm)
fig.colorbar(j_q_plot)

rl_a, rl_b, rl_c = magint_test.recip_lats

ax.quiver(0, 0, 0, rl_a[0], rl_a[1], rl_a[2], pivot='tail', color='r')
ax.quiver(0, 0, 0, rl_b[0], rl_b[1], rl_b[2], pivot='tail', color='g')
ax.quiver(0, 0, 0, rl_c[0], rl_c[1], rl_c[2], pivot='tail', color='b')

ax.set_title('J(q)')
ax.set_xlabel('qx')
ax.set_ylabel('qy')
ax.set_zlabel('qz')
plt.show()

<IPython.core.display.Javascript object>

In [11]:
print('From BZ recreation')
print(f'Jq = {np.max(magint_test.j_q[:,-1])}')
idx = np.argmax(magint_test.j_q[:,-1], axis = None)
qmax = magint_test.q_vecs[idx]
print(f'q = {qmax}')
print(f'q = {qmax @ np.linalg.inv(magint_test.recip_lats)} (rlu)')
print(f'q = {qmax * np.array([1 / qa, 1 / qb, 1 / qc])} (units of 2pi/a)')

From BZ recreation
Jq = 141.37822901033488
q = [-1.14196154  0.62711063  0.        ]
q = [-0.29651174  0.29651174  0.        ] (rlu)
q = [-0.57644729  0.31655727  0.        ] (units of 2pi/a)


In [12]:
# visualize J(q) at qz=0

qa = 2 * np.pi / a
qb = 2 * np.pi / b
qc = 2 * np.pi / c

qx = np.linspace(-0.5, 0.5, 101)
qy = np.linspace(-0.5, 0.5, 101)
qx, qy = np.meshgrid(qx, qy)
qz = 0.0


j_qs = np.zeros((magint_test.N, 1) + qx.shape)
u_qs = np.zeros((magint_test.N, magint_test.N, 1) + qx.shape)
for i in range(qx.shape[0]):
    for j in range(qx.shape[1]):
        qv = np.array([qx[i,j], qy[i,j], qz]) @ magint_test.recip_lats
        j_qs[:,:,i,j], u_qs[:,:,:,i,j] = magint_test.calc_j_q(qv, return_eig_vecs = True)
        
j_qs = j_qs.reshape((magint_test.N,) + qx.shape)
u_qs = u_qs.reshape((magint_test.N,magint_test.N,) + qx.shape)

tidx = 5

for k in range(magint_test.N):
    fig, ax = plt.subplots(subplot_kw = {"projection": "3d"})
    ax.plot_surface(qx, qy, j_qs[k])
    ax.set_zlim(-150, 150)
    ax.set_title('J(q)')
    ax.set_xlabel('qx')
    ax.set_ylabel('qy')
    plt.show()

<IPython.core.display.Javascript object>

In [13]:
print(f'From qz = {qz} slice')
print(f'Jq = {np.max(j_qs)}')
idx = np.unravel_index(np.argmax(j_qs, axis=None), j_qs.shape)
qmax = np.array([qx[idx[1:]], qy[idx[1:]], qz]) @ magint_test.recip_lats
print(f'q = {qmax}')
print(f'q = {qmax @ np.linalg.inv(magint_test.recip_lats)} (rlu)')
print(f'q = {qmax * np.array([1 / qa, 1 / qb, 1 / qc])} (units of 2pi/a)')

From qz = 0.0 slice
Jq = 142.5707895607157
q = [ 1.09181896 -0.61333855  0.        ]
q = [ 0.28 -0.29  0.  ] (rlu)
q = [ 0.55113597 -0.3096053   0.        ] (units of 2pi/a)


In [14]:
qxys = np.linspace(-0.5, 0.5, 1001)
jqxy = np.array([magint_test.calc_j_q(qxy * np.array([1, -1, 0]) @ magint_test.recip_lats)[0] for qxy in qxys])

fig, ax = plt.subplots()
ax.plot(qxys, jqxy)
ax.set_xlabel('qx (rlu)')
ax.set_ylabel('J(qx, qy)')
ax.set_title('qx = -qy')
plt.show()

<IPython.core.display.Javascript object>

In [15]:
qxys = np.linspace(-0.5, 0.5, 1001)
jqxy = np.array([magint_test.calc_j_q(np.array([qxy, qxy - 2 * 0.257174, 0]) @ magint_test.recip_lats)[0]
                 for qxy in qxys])

fig, ax = plt.subplots()
ax.plot(qxys,jqxy)
ax.set_xlabel('qx (rlu)')
ax.set_ylabel('J(qx, qy)')
ax.set_title('qy = qx - 2 * 0.257174')
plt.show()

<IPython.core.display.Javascript object>

Maxima near:
(1/4, -1/4, 0)
(-1/4, 1/4, 0)

In [16]:
# Calculate correlation parameters

r_max = 5.0
n_ord_vec = 2
calc_orf = True
dim = 1

xis_list = np.zeros((len(j_ijs), len(temps)))
lmop_list = np.zeros((len(j_ijs), len(temps)))
spin_squared_list = np.zeros((len(j_ijs)))

for i, j_ij in enumerate(j_ijs):
    
    j1, j2 = j_ij

    j_ratio = j2 / j1
    kvecx = 0.5 + (np.arcsin(j_ratio * np.sqrt(1 - 0.25 * j_ratio**2)) - np.arccos(-0.5 * j_ratio)) / (2 * np.pi)
    kvecy = -np.arccos(-0.5 * j_ratio) / (2 * np.pi)
    
    print(kvecx, kvecy)
    
    def calc_corr_parameters(spin_sq):
        
        magint = maginteractions.MagInteractions(magstruc=magstruc, temperatures=temps, j_ij=j_ij,
                                                 spin_squared=spin_sq, r_max=r_max, spin_dim=3)
        magint.make_interaction_matrix()
        magint.filter_spin_positions()
        ord_vec = np.array([kvecx, kvecy, 0]) @ magint.recip_lats

        magint.calculate_correlation_parameters(ord_vec=ord_vec, n_ord_vec=n_ord_vec, calc_orf=True,
                                                dim=dim, sum_rule_orf=False, n_bz_vecs=4, isotropic=False)
        xis = np.sort(np.linalg.eigvals(magint.damping_matrices), axis=1)[:,0]**(-1/2)
        
        return xis, magint.lmops
        
    
    def residuals(theta):
        spin_sq = theta[0]
        
        xis, lmops = calc_corr_parameters(theta[0])
        xis = xis[high_idx:]
        lmops = lmops
        
        res = np.concatenate([xis,lmops]) - np.concatenate([xi_high_data, lmop_data])
        weights = 1/np.concatenate([xi_high_err,lmop_err])
        res *= weights
        res = np.nan_to_num(res,nan=1e4)
        return res
    
    bounds = (0, 6)
    theta = [4]
    result = least_squares(residuals, theta, method='dogbox', bounds=bounds)
    
    print(result.cost)
    
    xis, lmops = calc_corr_parameters(result.x[0])
    xis_list[i] = xis
    lmop_list[i] = lmops
    spin_squared_list[i] = result.x[0]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex = True, figsize = (6, 8))

    ax1.scatter(data_temps, xi_data, c = 'black')
    ax1.errorbar(data_temps, xi_data, yerr = xi_err, fmt = 'none', c = 'black')
    ax1.plot(temps, xis)
    ax1.set_ylim(0, 50)
    ax1.set_ylabel(r'$\xi$ ($\AA$)')
    ax1.set_title(j_ij_names[i])

    ax2.scatter(data_temps, lmop_data, c = 'black')
    ax2.errorbar(data_temps, lmop_data, yerr = lmop_err,fmt = 'none', c = 'black')
    ax2.plot(temps, lmops)

    ax2.set_ylim(0, 5)
    ax2.set_xlabel('T (K)')
    ax2.set_ylabel(r'lmop ($\mu_B$)')

    plt.show()

0.2853028694288763 -0.2853028694288762


/home/edison/anaconda3/envs/diffpy/lib/python3.7/site-packages/diffpy/mpdf/magstructure.py:987: RuntimeWarning: invalid value encountered in true_divide
  distanceVecsN = distanceVecs/np.apply_along_axis(np.linalg.norm,1,distanceVecs)[:,np.newaxis]


29.35935497249447


<IPython.core.display.Javascript object>

0.2599536713846443 -0.25995367138464437
33.428357056302445


...

<IPython.core.display.Javascript object>

0.25812367719368956 -0.25812367719368956
38.16909430856951


<IPython.core.display.Javascript object>

0.25717745055007046 -0.2571774505500705
32.33027005977242


<IPython.core.display.Javascript object>

In [17]:
print(spin_squared_list)

[2.3580628  2.47320519 1.86735153 2.91151748 2.44817682 2.49147756]


In [18]:
magint = maginteractions.MagInteractions(magstruc=magstruc, temperatures=temps, j_ij=j_ij_zorko,
                                                 spin_squared=spin_squared_list[0], r_max=r_max, spin_dim=3)
magint.make_interaction_matrix()
magint.filter_spin_positions()
magint.calculate_orf_reciprocal_space(n_bz_vecs = 4)
recip_orfs = magint.orfs.copy()

ord_vec = np.array([kvecx, kvecy, 0]) @ magint.recip_lats

magint.calculate_correlation_parameters(ord_vec=ord_vec, n_ord_vec=n_ord_vec, calc_orf=True,
                                        dim=dim, sum_rule_orf=False, n_bz_vecs=4, isotropic=False)

In [19]:
# Visualize ORF

fig, ax = plt.subplots()
ax.plot(temps, magint.orfs, label = 'real-space orfs')
ax.plot(temps, recip_orfs, label = 'recip-space orfs')
# ax.plot(temps, experimental_orfs, label = 'experimental orfs')
ax.plot(temps, np.max(magint.j_q) - 3 * temps / spin_squared_list[-1], '--', label = 'min orf')
ax.plot(temps, np.zeros_like(temps), label = 'MF ORF')
ax.set_ylim(-50, 200)
ax.legend()
plt.show()

<IPython.core.display.Javascript object>

In [20]:
chi_0 = magint.calc_chi_0()
orf_sum_real = np.array([np.sum((1 - chi_0[i] * (magint.j_q - magint.orfs[i]))**(-1))
                         for i in range(len(temps))])
orf_sum_recip = np.array([np.sum((1 - chi_0[i] * (magint.j_q - recip_orfs[i]))**(-1))
                          for i in range(len(temps))])
orf_sum_max = np.array([np.sum((1 - chi_0[i] * (magint.j_q - np.max(magint.j_q)))**(-1))
                        for i in range(len(temps))])

fig, ax = plt.subplots()
ax.plot(temps, orf_sum_real / (magint.N * len(magint.q_vecs)), label='real-space orf_calc')
ax.plot(temps, orf_sum_recip / (magint.N * len(magint.q_vecs)), label='recip-space orf_calc')
ax.plot(temps, orf_sum_max / (magint.N * len(magint.q_vecs)), label='max orf')
ax.set_xlabel('T (K)')
ax.set_ylabel('<S(0)S(0)>/S(S+1)')
ax.set_title('NaMnO2')
ax.legend()
ax.set_ylim(0, 2)
plt.show()

<IPython.core.display.Javascript object>

In [21]:
# Visualize correlations

mPDF_tidx = 10

mstruc = magstruc.copy()
mstruc.dampingMat = magint.damping_matrices[mPDF_tidx]
mstruc.spins = mstruc.generateScaledSpins()

distances = np.apply_along_axis(np.linalg.norm, 1, mstruc.atoms)
viewmask = distances < 7
mstruc.visualize(mstruc.atoms[viewmask], mstruc.spins[viewmask], showcrystalaxes=True)

<IPython.core.display.Javascript object>

# Fit

In [22]:
def residuals(theta):
    
    j1, j2, spin_sq = theta
    j_ij = [j1, j2]
    
    j_ratio = j2 / j1
    kvecx = 0.5 + (np.arcsin(j_ratio * np.sqrt(1 - 0.25 * j_ratio**2)) - np.arccos(-0.5 * j_ratio)) / (2 * np.pi)
    kvecy = -np.arccos(-0.5 * j_ratio) / (2 * np.pi)
    
    magint = maginteractions.MagInteractions(magstruc=magstruc, temperatures=temps, j_ij=j_ij,
                                                 spin_squared=spin_sq, r_max=r_max, spin_dim=3)
    magint.make_interaction_matrix()
    magint.filter_spin_positions()

    ord_vec = np.array([kvecx, kvecy, 0]) @ magint.recip_lats

    magint.calculate_correlation_parameters(ord_vec=ord_vec, n_ord_vec=n_ord_vec, calc_orf=True,
                                            dim=dim, sum_rule_orf=False, n_bz_vecs=4, isotropic=False)
    xis = np.sort(np.linalg.eigvals(magint.damping_matrices), axis=1)[:,0]**(-1/2)
    lmops = magint.lmops
    
    xis = xis[high_idx:]
    lmops = lmops
    
    res = np.concatenate([xis,lmops]) - np.concatenate([xi_high_data, lmop_data])
    weights = 1/np.concatenate([xi_high_err,lmop_err])
    res *= weights
    res = np.nan_to_num(res,nan=1e4)
    return res

bounds = ([-np.inf, -np.inf, 0],[0, 0, np.inf])
theta = [-65.0, -25.0, 2.5]

result = least_squares(residuals, theta, method='dogbox', bounds=bounds)

In [23]:
j1, j2, spin_sq = result.x
j_ij = [j1, j2]

j_ratio = j2 / j1
kvecx = 0.5 + (np.arcsin(j_ratio * np.sqrt(1 - 0.25 * j_ratio**2)) - np.arccos(-0.5 * j_ratio)) / (2 * np.pi)
kvecy = -np.arccos(-0.5 * j_ratio) / (2 * np.pi)

magint_fit = maginteractions.MagInteractions(magstruc=magstruc, temperatures=temps, j_ij=j_ij,
                                             spin_squared=spin_sq, r_max=r_max, spin_dim=3)
magint_fit.make_interaction_matrix()
magint_fit.filter_spin_positions()

ord_vec = np.array([kvecx, kvecy, 0]) @ magint_fit.recip_lats

magint_fit.calculate_correlation_parameters(ord_vec=ord_vec, n_ord_vec=n_ord_vec, calc_orf=True,
                                        dim=dim, sum_rule_orf=False, n_bz_vecs=4, isotropic=False)
xis_fit = np.sort(np.linalg.eigvals(magint_fit.damping_matrices), axis=1)[:,0]**(-1/2)
lmops_fit = magint_fit.lmops

In [24]:
print(result.jac.T @ result.jac)
print(f'Conditional Variance = {np.diag(result.jac.T @ result.jac)**(-1/2)}')
print(f'Variance = {np.diag(np.linalg.inv(result.jac.T @ result.jac))**(1/2)}')

[[  14.53764222   57.74191427 -183.41494315]
 [  57.74191427  234.59437292 -742.91750413]
 [-183.41494315 -742.91750413 2528.49276207]]
Conditional Variance = [0.26227265 0.06528918 0.01988699]
Variance = [1.75800033 0.48331233 0.07562409]


In [25]:
for i, j in enumerate(result.x):
    if i != 2:
        print(f'J{i+1}={j}')

print(f'S^2={result.x[2]}')

print(f'Cost = {result.cost}')

J1=-65.4246440904384
J2=-23.916166337977558
S^2=2.417672515723801
Cost = 28.19728844755904


In [26]:
# Calculate mPDF

rdat, mpdf_obs, mpdf_calc, _ = np.loadtxt('NaMnO2Data/mPDFfitSub280_50.0K.txt', skiprows=2, delimiter=' ').T

mpdf_tidx = 10

paraScale = 0.01

mstruc = magstruc.copy()

mstruc.dampingMat = magint_fit.damping_matrices[mpdf_tidx]
mc = MPDFcalculator(mstruc, qmax=35.0, qdamp=0.03, rmin=1.5)
mc.paraScale = paraScale
mc.ordScale = magutils.calculate_ordered_scale(mstruc,lmops_fit[mpdf_tidx])
rcalc, dcalc = mc.calc(correlationMethod='full', normalized=False, both=False)

In [27]:
from matplotlib.gridspec import GridSpec

labelsize=10
ticksize=10

colors = ['tab:purple', 'tab:green', 'tab:blue', 'tab:orange', 'tab:red', 'tab:brown']

fig = plt.figure(layout="constrained", figsize=(12, 2.4))
gs = GridSpec(1, 4, figure=fig)
ax_mpdf = fig.add_subplot(gs[0, 0:2])
ax1 = fig.add_subplot(gs[0, 2])
ax2 = fig.add_subplot(gs[0, 3])

ax_mpdf.plot(rdat, mpdf_obs, 'bo', label="mPDF(r) data",
             markerfacecolor='none', markeredgecolor='black')
ax_mpdf.plot(rdat, mpdf_calc, 'r-', lw=2.5, label="mPDF(r) atomistic fit")
ax_mpdf.plot(rcalc, dcalc, 'y-', lw=2.5, label='mPDF(r) from exchange fits')
ax_mpdf.set_title('NaMnO2')
ax_mpdf.set_xlim(1.5, 20)
ax_mpdf.set_xlabel(r'$r$ ($\AA$)', fontsize=labelsize)
ax_mpdf.set_ylabel(r'T=50K mPDF$(r)$ (a.u.)',fontsize=labelsize)

ax1.scatter(data_temps, xi_data, c='black', label=r'data')
ax1.errorbar(data_temps, xi_data, yerr=xi_err, fmt='none', c='black')
ax1.plot(temps, xis_fit, '-', c='y', lw=3, label='fit', alpha=1)
for i in range(2):
    ax1.plot(temps, xis_list[i], '--', alpha=1, c=colors[i], lw=2, label=j_ij_names[i])
ax1.set_xlim(0, 275)
ax1.set_ylim(0, 40)
ax1.set_xlabel('T (K)', fontsize=labelsize)
ax1.set_ylabel(r'$\xi$ ($\AA$)', fontsize=labelsize)
ax1.tick_params(axis = 'x', labelsize=ticksize)
ax1.tick_params(axis = 'y', labelsize=ticksize)

ax2.scatter(data_temps, lmop_data, c = 'black')
ax2.errorbar(data_temps, lmop_data, yerr = lmop_err, fmt = 'none', c = 'black')
ax2.plot(temps, lmops_fit, c = 'y', lw=3, label='fit', alpha=1)
for i in range(2):
    ax2.plot(temps, lmop_list[i], '--', lw=2, alpha=1, c=colors[i], label=j_ij_names[i])
ax2.set_xlim(0, 275)
ax2.set_ylim(0, 4)
ax2.set_xlabel('T (K)', fontsize = labelsize)
ax2.set_ylabel(r'$m_{\mathrm{SRO}}$ ($\mu_B$)', fontsize=labelsize)
ax2.legend(fontsize=8)
ax2.tick_params(axis='x', labelsize=ticksize)
ax2.tick_params(axis='y', labelsize=ticksize)

plt.savefig('figures/NaMnO2Fits.png', dpi=1200)
plt.show()

<IPython.core.display.Javascript object>

# Which temperatures are most important?

In [ ]:
### Optimal Experiment Design
from __future__ import annotations
from scipy.optimize._numdiff import approx_derivative
from numpy.typing import NDArray

import warnings
import numpy as np
import cvxpy as cp

def model(x, theta):
    j1, j2 = theta
    j_ij = [j1, j2]
    
    j_ratio = j2 / j1
    kvecx = 0.5 + (np.arcsin(j_ratio * np.sqrt(1 - 0.25 * j_ratio**2)) - np.arccos(-0.5 * j_ratio)) / (2 * np.pi)
    kvecy = -np.arccos(-0.5 * j_ratio) / (2 * np.pi)
    
    magint = maginteractions.MagInteractions(magstruc=magstruc, temperatures=x, j_ij=j_ij,
                                                 spin_squared=spin_sq, r_max=r_max, spin_dim=3)
    magint.make_interaction_matrix()
    magint.filter_spin_positions()

    ord_vec = np.array([kvecx, kvecy, 0]) @ magint.recip_lats

    magint.calculate_correlation_parameters(ord_vec=ord_vec, n_ord_vec=n_ord_vec, calc_orf=True,
                                            dim=dim, sum_rule_orf=False, n_bz_vecs=4, isotropic=False)
    xis = np.sort(np.linalg.eigvals(magint.damping_matrices), axis=1)[:,0]**(-1/2)
    lmops = magint.lmops
    return np.concatenate([xis, lmops])

def jacobian(
    theta: NDArray[np.float64],
    x: NDArray[np.float64],
    model_fn,
) -> NDArray[np.float64]:
    """
    Numerical Jacobian via central differences with adaptive step size.

    Parameters
    ----------
    theta : ndarray, shape (M,)
        Parameters — differentiation variable.
    x : ndarray, shape (N,)
        Fixed inputs, passed through to model_fn via closure/args.
    model_fn : Callable[[ndarray], ndarray]
        Must take theta ONLY and return y of shape (N,); wrap with a lambda
        to freeze x.

    Returns
    -------
    J : ndarray, shape (N, M)
    """
    return approx_derivative(
        model_fn, theta, method="3-point", rel_step=None  # None -> auto-selects optimal h
    )

def minimize_l1_norm_psd_constraint(
    A_stack: np.ndarray,
    B: np.ndarray,
    strict_margin: float = 0.0,
    solver: str = 'SCS',
    eps: float = 1e-6,
    max_iters: int = 100000,
) -> tuple[np.ndarray, float, str]:
    """Minimize sum(x) subject to x_i > 0 and (sum_i x_i * A_i - B) is PSD.

    Parameters
    ----------
    A_stack : np.ndarray, shape (m, n, n)
        Stack of m constraint matrices, A_stack[i] = A_i. Each A_i need
        NOT be symmetric a priori, but see note in Critical Analysis --
        only the symmetric part of the combination is physically
        meaningful for a PSD constraint, and asymmetric A_i most likely
        indicates a modeling bug.
    B : np.ndarray, shape (n, n)
        Target matrix; constraint is (sum_i x_i * A_i - B) >> 0.
    strict_margin : float, default 1e-9
        Proxy for strict positivity x_i > 0.
    solver : str, default cp.CLARABEL
        SDP-capable CVXPY solver. Alternatives: cp.SCS (older, more
        widely available in older CVXPY installs), cp.MOSEK (commercial,
        much faster/more robust for larger n, requires a license).

    Returns
    -------
    x_opt : np.ndarray, shape (m,)
        Optimal weights.
    optimal_value : float
        Minimal value of sum(x) (== ||x||_1 since x > 0).
    status : str
        CVXPY problem status string (e.g. "optimal", "infeasible").

    Raises
    ------
    ValueError
        If shapes are inconsistent, or if A_i / B are not square.
    RuntimeError
        If the solver does not report an optimal solution.
    """
    A_stack = np.asarray(A_stack, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)

    if A_stack.ndim != 3 or A_stack.shape[1] != A_stack.shape[2]:
        raise ValueError(f"A_stack must have shape (m, n, n); got {A_stack.shape}.")
    m, n, _ = A_stack.shape
    if B.shape != (n, n):
        raise ValueError(f"B must have shape ({n}, {n}); got {B.shape}.")

    # PSD constraints are only meaningful for symmetric matrices (eigenvalues
    # of a non-symmetric real matrix can be complex, so "PSD" is undefined).
    # If your A_i, B aren't already symmetric, this symmetrizes them via
    # (M + M.T)/2, which is the standard convention -- see Critical Analysis.
    B_sym = 0.5 * (B + B.T)
    A_stack_sym = 0.5 * (A_stack + A_stack.transpose(0, 2, 1))

    x = cp.Variable(m, pos=True)  # pos=True enforces x_i > 0 approx. (see notes)

    # Build the weighted sum sum_i x_i * A_i as a CVXPY expression.
    # cp.sum with a list comprehension is the standard idiom for this;
    # there is no vectorized "einsum" equivalent in CVXPY's expression
    # graph API, since each term must remain a traceable affine atom.
    weighted_sum = cp.sum([x[i] * A_stack_sym[i] for i in range(m)])

    constraints = [
        x >= strict_margin,           # numerical proxy for x_i > 0
        weighted_sum - B_sym >> 0,    # the LMI / PSD constraint
    ]

    objective = cp.Minimize(cp.sum(x))  # == ||x||_1 since x > 0
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=solver, eps=eps, max_iters=max_iters)

    if problem.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"SDP did not solve to optimality: status={problem.status}")
        
    if problem.status == "optimal_inaccurate":
        warnings.warn(f"Iteration solve was inaccurate (status={problem.status}); "
                       "consider tightening SCS tolerance or checking convergence stability.")
    elif problem.status != "optimal":
        raise RuntimeError(f"SDP failed: status={problem.status}")

    print(f"Number of iterations: {problem.solver_stats.num_iters}")

    return x.value, problem.value, problem.status

# Initialize

x = data_temps_high
theta_0 = np.array([j1, j2])
sigma = np.diag(0.1 * theta_0) ** 2
w_opt = np.zeros_like(x)
max_iter = 1
tol = 0.05

fim_fm_array = np.zeros((len(x), len(theta_0), len(theta_0)))

for i in range(max_iter):
    fim_g = np.linalg.inv(sigma)
    
    for j in range(len(x)):
        x_m = np.array([x[j]])
        j_fm = jacobian(theta_0, x_m, lambda th: model(x_m, th))
        fim_fm = j_fm.T @ j_fm
        
        fim_fm_array[j] = fim_fm
    
    w, opt_val, status = minimize_l1_norm_psd_constraint(fim_fm_array, fim_g, eps=1e-10,
                                                             max_iters=1000000000)
    print(f"Status: {status}")
    print(f"Minimal 1-norm: {opt_val:.6f}")

    # Verify: eigenvalues of (sum_m w_m fim_fm - fim_g) should all be >= 0
    reconstructed = np.einsum("i,ijk->jk", w_opt, fim_fm_array)
    eigvals = np.linalg.eigvalsh(reconstructed - fim_g)
    print(f"Min eigenvalue of (A - B) (should be >= -tol): {eigvals.min():.6e}")
    
    w_old = w_opt.copy()
    w_opt = np.maximum(w, w_opt)
    delta_w = np.sum(np.abs(w_opt - w_old))
    
    print(f"\nFor theta_0 = {theta_0}")
    print(f"Optimal w: {w_opt}")
    print(f"delta w = {delta_w}\n")
        
    if delta_w < tol:
        break
    
    def residuals(theta):
        f = model(x, theta)
        y = np.concatenate([xi_high_data, lmop_high_data])

        res = y - f
        weights = np.sqrt(np.concatenate([w_opt, w_opt]))

        return weights * res
    
    result = least_squares(residuals, theta_0, method = 'lm')
    theta_0 = result.x
    
print(f'Necessary temperatures: {data_temps_high[np.where(w_opt > 1e-8)]}')

Number of iterations: 5200
Status: optimal
Minimal 1-norm: 1660.498376
Min eigenvalue of (A - B) (should be >= -tol): -1.748304e-01

For theta_0 = [-65.42464409 -23.91616634]
Optimal w: [767.26874282   0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.         893.22963354   0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.        ]
delta w = 1660.498376361582



In [ ]:
def residuals_necessary_temps(theta):
    idxs_to_keep = np.where(w_opt >= 1e-9)
    x = data_temps_high[idxs_to_keep]
    f = model(x, theta)
    y = np.concatenate([xi_high_data[idxs_to_keep], lmop_high_data[idxs_to_keep]])
    
    res = y - f
    weights = 1 / np.concatenate([xi_high_err[idxs_to_keep], lmop_high_err[idxs_to_keep]])
    
    return np.asarray(weights * res, dtype=float).flatten()

theta = [-65.4, -23.9]
result = least_squares(residuals_necessary_temps, theta, method="trf", diff_step=1e-9)

print(result.x)

In [ ]:
fit_output = model(data_temps, result.x)
xis_fit = fit_output[:len(data_temps)]
lmops_fit = fit_output[len(data_temps):]

plt.figure()
plt.plot(data_temps, xis_fit)
plt.scatter(data_temps, xi_data, c = 'black', label = r'data')
plt.errorbar(data_temps,xi_data, yerr = xi_err, fmt = 'none', c = 'black')
plt.ylim(0,50)
plt.show()

plt.figure()
plt.plot(data_temps, lmops_fit)
plt.scatter(data_temps, lmop_data, c = 'black')
plt.errorbar(data_temps, lmop_data, yerr = lmop_err, fmt = 'none', c = 'black')
plt.ylim(0,5)
plt.show()

# Cost Surface

In [ ]:
def cost(theta, spin_sq=2.4):    
    j1, j2 = theta
    j_ij = [j1, j2]
    
    if j2 < j1 or j1 == 0:
        return np.inf
    
    j_ratio = j2 / j1
    kvecx = 0.5 + (np.arcsin(j_ratio * np.sqrt(1 - 0.25 * j_ratio**2)) - np.arccos(-0.5 * j_ratio)) / (2 * np.pi)
    kvecy = -np.arccos(-0.5 * j_ratio) / (2 * np.pi)
    
    magint = maginteractions.MagInteractions(magstruc=magstruc, temperatures=temps, j_ij=j_ij,
                                                 spin_squared=spin_sq, r_max=r_max, spin_dim=3)
    magint.make_interaction_matrix()
    magint.filter_spin_positions()

    ord_vec = np.array([kvecx, kvecy, 0]) @ magint.recip_lats

    magint.calculate_correlation_parameters(ord_vec=ord_vec, n_ord_vec=n_ord_vec, calc_orf=True,
                                            dim=dim, sum_rule_orf=False, n_bz_vecs=4, isotropic=False)
    xis = np.sort(np.linalg.eigvals(magint.damping_matrices), axis=1)[:,0]**(-1/2)
    lmops = magint.lmops
    
    xis = xis[high_idx:]
    lmops = lmops
    
    res = np.concatenate([xis,lmops]) - np.concatenate([xi_high_data, lmop_data])
    weights = 1/np.concatenate([xi_high_err,lmop_err])
    res *= weights
    res = np.nan_to_num(res,nan=1e4)

    cost = 0.5 * np.sum(res ** 2)
    return cost

In [ ]:
j1s = np.arange(-70, -60)
j2s = np.arange(-30, -20)

cost_surface = np.zeros((len(j1s), len(j2s)))

for i in range(len(j2s)):
    for j in range(len(j1s)):
        print(j1s[j], j2s[i])
        cost_surface[j, i] = cost([j1s[j], j2s[i]])

In [ ]:
plt.figure()
plt.imshow(
    cost_surface,
    extent=[j2s.min(), j2s.max(), j1s.min(), j1s.max()],
    vmax=36,
    origin='lower',
    aspect='auto',
    cmap='YlOrBr',
)
plt.colorbar(label='Cost')
plt.xlabel('J2')
plt.ylabel('J1')
plt.title('Cost Surface')
plt.show()

In [ ]:
np.min(cost_surface)